# SpeakAI-Eval — Inference (Run Only)

Notebook này chỉ cần gọi lên và chạy. Đảm bảo bạn đã chạy **Setup Notebook** trước đó.
Mã nguồn, models và HF cache sẽ được đọc thẳng từ Kaggle Dataset.

In [ ]:
# Cài đặt thư viện
import subprocess, sys, os
try:
    import numpy
    np_ver = numpy.__version__
except: np_ver = '2.0.2'
print('Installing Rust (required for DeepFilterNet on Python 3.12)...')
os.system("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y")
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ.get('PATH', '')}"
pkgs = [
    'torch>=2.1.0', 'torchaudio>=2.1.0', 'torch-geometric>=2.4.0',
    'transformers>=4.36.0', 'peft>=0.7.0', 'pyyaml>=6.0.1',
    'numpy>=1.24.0', 'soundfile>=0.12.1', 'nltk>=3.8.1',
    'python-dotenv>=1.0.0', 'deepfilternet', 'addict', 'modelscope',
    'speechbrain>=1.0.0', 'huggingface_hub>=0.23.0',
    'accelerate>=0.26.0', 'fastapi', 'uvicorn', 'python-multipart',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', '-q', f'numpy=={np_ver}'], check=True)
print('Libraries installed!')
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('Downloading cloudflared...')
    subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared', shell=True, check=True)
    subprocess.run('chmod +x /usr/local/bin/cloudflared', shell=True, check=True)


---
## Khởi tạo Pipeline từ Kaggle Input

In [ ]:
import sys, os
SOURCE_DIR = None
MODEL_DIR = None
PRONUNCIATION_PATH = None
for search_dir in ['/kaggle/working', '/kaggle/input']:
    for root, dirs, files in os.walk(search_dir):
        if 'pretrained_models' in dirs and 'wavlm-large' in os.listdir(os.path.join(root, 'pretrained_models')):
            if MODEL_DIR is None: MODEL_DIR = root
        if 'infer' in dirs and 'speaker-diarize' in dirs:
            if SOURCE_DIR is None: SOURCE_DIR = root
        if 'pronunciation.pt' in files:
            if PRONUNCIATION_PATH is None: PRONUNCIATION_PATH = os.path.join(root, 'pronunciation.pt')
if not MODEL_DIR:
    raise FileNotFoundError("Không tìm thấy thư mục 'pretrained_models/wavlm-large' trong dataset model.")
if not SOURCE_DIR:
    raise FileNotFoundError("Không tìm thấy source code (infer, speaker-diarize) trong dataset setup.")
if not PRONUNCIATION_PATH:
    raise FileNotFoundError("Không tìm thấy file 'pronunciation.pt' (mô hình speechocean) trong bất kỳ dataset nào.")
sys.path.insert(0, SOURCE_DIR)
sys.path.insert(0, f'{SOURCE_DIR}/speaker-diarize')
os.chdir(SOURCE_DIR)

# Lưu cache HuggingFace vào MODEL_DIR (nếu có thể) hoặc /kaggle/working
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'

import torch, yaml
from infer.pipeline import SpeakingPipeline
from transformers import AutoModelForCausalLM, AutoTokenizer

print('Patching config dynamically...')
with open(f'{SOURCE_DIR}/configs/pronunciation.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
cfg['asr']['model_name'] = f'{MODEL_DIR}/pretrained_models/whisper-large-v3'
cfg['wavlm']['model_name'] = f'{MODEL_DIR}/pretrained_models/wavlm-large'
cfg['paths']['pronunciation_checkpoint'] = PRONUNCIATION_PATH
os.makedirs('/kaggle/working/tmp_configs', exist_ok=True)
tmp_config = '/kaggle/working/tmp_configs/pronunciation.yaml'
with open(tmp_config, 'w') as f:
    yaml.dump(cfg, f)

print('Patching transcribe.py dynamically...')
import infer.transcribe
def patched_load_asr_config():
    with open(tmp_config, encoding='utf-8') as f:
        return yaml.safe_load(f).get('asr') or {}
infer.transcribe._load_asr_config = patched_load_asr_config

print('Loading Pipeline models on GPU 1 (cuda:1)...')
pipeline = SpeakingPipeline(config_path=tmp_config, device='cuda:1')
print('Pipeline ready!')

print('Loading Registration Model on GPU 0 (cuda:0)...')
from speaker_diarize.embedding import ERes2NetEmbedder
extract_embedder = ERes2NetEmbedder(device='cuda:0')
print('Registration Embedder ready!')

model_name = f'{MODEL_DIR}/pretrained_models/Qwen2.5-3B-Instruct'
print(f'Loading LLM from Kaggle: {model_name}')
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map='cuda:1'
)
print(f'LLM Ready on {llm_model.device}!')


---
## Hàm Sinh Feedback Tổng Hợp bằng Qwen LLM

In [ ]:
def generate_turn_feedback(teacher_text, student_text, score, errors):
    bad_words = [w for w in errors.get('words', []) if w.get('score', 10) < 7.0]
    bad_ph = [p for p in errors.get('phonemes', []) if p.get('score', 10) < 7.0]
    bad_words_str = ', '.join([f"{w['word']}" for w in bad_words])
    bad_ph_str = ', '.join([f"{p['phoneme']}" for p in bad_ph])
    if bad_words_str or bad_ph_str:
        return f"⚠️ Cần cải thiện: Phát âm yếu các từ [{bad_words_str}]. Sai âm vị [{bad_ph_str}]."
    return ""

def generate_overall_summary(result):
    student_sents = result.get('student', {}).get('sentences', [])
    if not student_sents:
        return 'Không có dữ liệu.'
    
    total_acc = sum(s.get('scores', {}).get('accuracy', 0) for s in student_sents) / len(student_sents)
    total_flu = sum(s.get('scores', {}).get('fluency', 0) for s in student_sents) / len(student_sents)
    total_pro = sum(s.get('scores', {}).get('prosodic', 0) for s in student_sents) / len(student_sents)
    overall_total = (total_acc + total_flu + total_pro) / 3
    
    prompt = f"""Dựa trên tổng điểm của cuộc hội thoại:\nTổng quan: {overall_total:.1f}/10\nChính xác (Accuracy): {total_acc:.1f}/10\nTrôi chảy (Fluency): {total_flu:.1f}/10\nNgữ điệu (Prosody): {total_pro:.1f}/10\n\nHãy viết tổng kết đánh giá chi tiết THEO ĐÚNG ĐỊNH DẠNG SAU (không dài dòng):\n🌟 Điểm mạnh:\n[1 câu nhận xét chung]\n🎯 Cần cải thiện:\n[1 điểm cần lưu ý nhất]\n💪 Gợi ý luyện tập:\n[1 gợi ý ngắn gọn]"""
    messages = [
        {'role': 'system', 'content': 'Bạn là giáo viên tiếng Anh. Đưa ra nhận xét cực kỳ ngắn gọn, đơn giản.'},
        {'role': 'user', 'content': prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors='pt').to(llm_model.device)
    
    generated_ids = llm_model.generate(**model_inputs, max_new_tokens=512, repetition_penalty=1.1)
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()


---
## Khởi chạy Backend API (FastAPI + Cloudflare Tunnel)

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form, BackgroundTasks
import uuid
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from fastapi.responses import JSONResponse
import uvicorn
import shutil
import json
import numpy as np
import subprocess
import time
import re
import os

# 1. Khởi động Cloudflare Quick Tunnel
def start_cloudflare_tunnel(port=8000):
    print('Starting Cloudflare Quick Tunnel...')
    cmd = f'cloudflared tunnel --url http://127.0.0.1:{port}'
        
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url = None
        
    for _ in range(20):
        line = process.stdout.readline()
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            break
        time.sleep(0.5)
    return url

PUBLIC_URL = start_cloudflare_tunnel(8000)
print('\n' + '='*80)
print(f'🚀 API IS LIVE AT: {PUBLIC_URL}')
print('=> COPY LINK NÀY VÀ DÁN VÀO CẤU HÌNH TRÊN WEBSITE CỦA BẠN!')
print('='*80 + '\n')

# 2. Khởi tạo FastAPI App
app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

# Phục vụ file audio tĩnh từ Colab để Website có thể nghe lại
os.makedirs('/tmp/SpeakAI_Audio', exist_ok=True)
app.mount('/audio', StaticFiles(directory='/tmp/SpeakAI_Audio'), name='audio')

tasks = {}

@app.post('/extract_embedding')
def extract_embedding_api(audio: UploadFile = File(...)):
    try:
        temp_id = str(uuid.uuid4())
        raw_audio_path = f'/tmp/SpeakAI_Audio/{temp_id}_raw_{audio.filename}'
        with open(raw_audio_path, 'wb') as f:
            shutil.copyfileobj(audio.file, f)
        
        # Convert to standard WAV using ffmpeg
        audio_path = f'/tmp/SpeakAI_Audio/{temp_id}_converted.wav'
        os.system(f'ffmpeg -y -i \"{raw_audio_path}\" -ar 16000 -ac 1 \"{audio_path}\" -loglevel quiet')
        
        # Tải audio
        from speaker_diarize.audio_io import load_audio
        waveform, sr = load_audio(audio_path)
        
        from speaker_diarize.denoise import denoise_with_deepfilternet, level_audio_to_target
        # Khử nhiễu & Cân bằng âm lượng
        waveform, sr = denoise_with_deepfilternet(waveform, sr)
        waveform = level_audio_to_target(waveform, sr)
            
        # Tính toán embedding bằng extract_embedder trên cuda:0
        emb = extract_embedder.embed(waveform, sr)
        
        # Xóa file tạm
        if os.path.exists(audio_path):
            os.remove(audio_path)
            
        return JSONResponse({'success': True, 'embedding': emb.tolist()})
    except Exception as e:
        import traceback
        traceback.print_exc()
        return JSONResponse({'success': False, 'error': str(e)}, status_code=500)

@app.post('/assess_start')
def assess_start_api(background_tasks: BackgroundTasks, audio: UploadFile = File(...), teacher_embeddings_json: str = Form(...), student_embeddings_json: str = Form(...), score_teacher: bool = Form(False), skip_feedback: bool = Form(False)):
    try:
        task_id = str(uuid.uuid4())
        tasks[task_id] = {'status': 'processing', 'step': 'Đang tải file âm thanh lên server...', 'result': None, 'llm_feedback': None}
        
        raw_conv_path = f'/tmp/SpeakAI_Audio/{task_id}_raw_{audio.filename}'
        with open(raw_conv_path, 'wb') as f:
            shutil.copyfileobj(audio.file, f)
        
        conv_path = f'/tmp/SpeakAI_Audio/{task_id}_converted.wav'
        os.system(f'ffmpeg -y -i \"{raw_conv_path}\" -ar 16000 -ac 1 \"{conv_path}\" -loglevel quiet')
        
        background_tasks.add_task(process_assessment, task_id, conv_path, teacher_embeddings_json, student_embeddings_json, score_teacher, skip_feedback)
        return JSONResponse({'success': True, 'task_id': task_id})
    except Exception as e:
        return JSONResponse({'success': False, 'error': str(e)}, status_code=500)

def process_assessment(task_id, conv_path, teacher_embeddings_json, student_embeddings_json, score_teacher, skip_feedback):
    try:
        tasks[task_id]['step'] = 'Đang phân tích embeddings...'
        
        # Parse Embeddings của Giáo viên
        t_emb_list = json.loads(teacher_embeddings_json)
        teacher_emb = np.array(t_emb_list, dtype=np.float32)
        if len(teacher_emb.shape) == 2:
            teacher_emb = np.mean(teacher_emb, axis=0)
        teacher_emb /= np.linalg.norm(teacher_emb)

        # Parse Embeddings của Học viên
        s_emb_list = json.loads(student_embeddings_json)
        student_emb = np.array(s_emb_list, dtype=np.float32)
        if len(student_emb.shape) == 2:
            student_emb = np.mean(student_emb, axis=0)
        student_emb /= np.linalg.norm(student_emb)
        
        # Gọi Pipeline
        tasks[task_id]['step'] = 'Đang tách lời (Diarization) & Phân tích phát âm...'
        raw_result = pipeline.assess_conversation(
            conv_path,
            teacher_embedding=teacher_emb,
            student_embedding=student_emb,
            score_teacher=score_teacher
        )
        
        def apply_penalty(node):
            if isinstance(node, dict):
                for k, v in node.items():
                    if k in ['accuracy', 'fluency', 'prosodic', 'score', 'completeness', 'total'] and isinstance(v, (int, float)) and v > 0:
                        node[k] = max(0.0, v - (10.0 - v) * 0.15)
                    else:
                        apply_penalty(v)
            elif isinstance(node, list):
                for item in node:
                    apply_penalty(item)
                    
        apply_penalty(raw_result)
        
        # Trích xuất file tổng hợp
        diar = raw_result.get('diarization', {})
        if raw_result.get('teacher') and diar.get('teacher'):
            raw_result['teacher']['full_audio'] = str(diar['teacher'])
        if raw_result.get('student') and diar.get('student'):
            raw_result['student']['full_audio'] = str(diar['student'])

        # Gọi LLM Feedback
        if skip_feedback:
            tasks[task_id]['step'] = 'Bỏ qua LLM Feedback...'
            llm_feedback = 'Không có phản hồi (bỏ qua bởi người dùng).'
        else:
            tasks[task_id]['step'] = 'Đang gọi LLM Qwen tạo Feedback cho từng lượt...'
            teacher_ctx = 'Không có'
            for turn in raw_result.get('dialogue', {}).get('turns', []):
                if turn['role'].upper() == 'TEACHER':
                    teacher_ctx = turn['transcript']
                elif turn['role'].upper() == 'STUDENT':
                    turn['llm_feedback'] = generate_turn_feedback(
                        teacher_text=teacher_ctx, 
                        student_text=turn['transcript'], 
                        score=turn.get('scores', {}).get('accuracy', 0), 
                        errors=turn.get('errors', {})
                    )
            
            tasks[task_id]['step'] = 'Đang gọi LLM Qwen tạo Feedback tổng hợp...'
            llm_feedback = generate_overall_summary(raw_result)
        
        # Chuyển đổi đường dẫn file cục bộ thành Public URL
        def convert_paths_to_urls(node):
            if isinstance(node, dict):
                for k, v in node.items():
                    if (k == 'audio' or k == 'full_audio') and isinstance(v, str) and v.startswith('/tmp/SpeakAI_Audio/'):
                        rel_path = v.replace('/tmp/SpeakAI_Audio/', '')
                        node[k] = f'{PUBLIC_URL}/audio/{rel_path}'
                    else:
                        convert_paths_to_urls(v)
            elif isinstance(node, list):
                for item in node:
                    convert_paths_to_urls(item)
                    
        convert_paths_to_urls(raw_result)
        
        # Xóa file audio tạm
        if os.path.exists(conv_path):
            os.remove(conv_path)
            
        tasks[task_id]['result'] = raw_result
        tasks[task_id]['llm_feedback'] = llm_feedback
        tasks[task_id]['status'] = 'completed'
    except Exception as e:
        print(f'API Error in task {task_id}: {e}')
        import traceback
        traceback.print_exc()
        tasks[task_id]['status'] = 'error'
        tasks[task_id]['error'] = str(e)

@app.get('/assess_status/{task_id}')
def assess_status(task_id: str):
    if task_id not in tasks:
        return JSONResponse({'success': False, 'error': 'Task not found'}, status_code=404)
    return JSONResponse({'success': True, 'data': tasks[task_id]})

# Khởi chạy Uvicorn
config = uvicorn.Config(app, host='0.0.0.0', port=8000)
server = uvicorn.Server(config)
await server.serve()
